# Phase 1 — Knowledge Distillation Training (Kaggle)

Trains a **Vietnamese student SentenceTransformer** via knowledge distillation from a multilingual teacher (`distiluse-base-multilingual-cased-v2`) on an English–Vietnamese parallel corpus.

**Datasets to attach before running:**

| Kaggle Dataset | Role |
|---|---|
| Your uploaded `LRL-IR` source zip | Project source code |
| `en_vi_translation.csv` dataset | Parallel EN–VI bitext |

**Key fixes applied vs the original notebook:**
- Learning rate lowered from `1e-4` → `2e-5` (was 10× too high, caused NaN weights)
- Gradient clipping added in `knowledge_distillation.py` (`max_norm=1.0`)
- NaN/Inf loss guard before `backward()` in `knowledge_distillation.py`

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers rank-bm25 py-vncorenlp POT python-dotenv scikit-learn

## 2. Clone / mount source code & set paths

Two options depending on how you attached the code:
- **Option A** (recommended): Upload LRL-IR as a Kaggle Dataset and mount it.
- **Option B**: `git clone` from GitHub (requires internet enabled).

In [ ]:
import os, sys, torch

# ── Option A: LRL-IR uploaded as a Kaggle Dataset ────────────────────────────
# The dataset slug is the lowercase name you gave it on Kaggle, e.g. 'lrl-ir-src'
# Kaggle mounts it at /kaggle/input/<slug>/<folder-name-in-zip>/
# Adjust the path below to match your upload.
PROJECT_DIR = "/kaggle/input/lrl-ir-src/LRL-IR/"   # ← adjust if needed

# ── Option B: git clone (comment out Option A above and uncomment below) ──────
# !git clone -b Trong https://github.com/sitrongtrang/LRL-IR.git /kaggle/working/LRL-IR
# PROJECT_DIR = "/kaggle/working/LRL-IR/"

# ── Writable output dir ───────────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/LRL-IR/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Add project to Python path & set PROJECT_DIR env var ─────────────────────
sys.path.insert(0, PROJECT_DIR)
os.environ["PROJECT_DIR"] = OUTPUT_DIR   # KnowledgeDistillation.train() reads this

# ── Remove stale VnCoreNLP models to avoid cache conflicts ───────────────────
!rm -rf /vncorenlp/models

print("PROJECT_DIR :", PROJECT_DIR, "→ exists:", os.path.isdir(PROJECT_DIR))
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("Python path :", sys.path[:3])

## 3. Prepare bitext CSV

The `KnowledgeDistillation` trainer reads a CSV with columns `source` (English) and `target` (Vietnamese).  
Adjust `input_csv_path` and `NUM_ROWS` to match your Kaggle dataset.

In [ ]:
import pandas as pd

# ── Source dataset ─────────────────────────────────────────────────────────────
# Kaggle dataset: thaideptrai3250/nlp-phase1  →  file: en_vi_translation.csv
input_csv_path = "/kaggle/input/datasets/thaideptrai3250/nlp-phase1/en_vi_translation.csv"

# How many rows to use for training (set None to use all rows)
NUM_ROWS = 8000

# ── Load & slice ───────────────────────────────────────────────────────────────
df = pd.read_csv(input_csv_path)
print(f"Full dataset: {len(df)} rows, columns: {list(df.columns)}")

# Ensure columns are named 'source' and 'target'
if "source" not in df.columns or "target" not in df.columns:
    raise ValueError(f"Expected columns 'source' and 'target', got {list(df.columns)}")

# Drop rows where either sentence is empty
df = df[["source", "target"]].dropna()
df = df[df["source"].str.strip() != ""]
df = df[df["target"].str.strip() != ""]

if NUM_ROWS is not None:
    df = df.head(NUM_ROWS).reset_index(drop=True)

# Save to writable output dir
BITEXT_CSV = os.path.join(OUTPUT_DIR, "bitext.csv")
df.to_csv(BITEXT_CSV, index=False)

print(f"Training rows   : {len(df)}")
print(f"Bitext CSV saved: {BITEXT_CSV}")
df.head(3)

## 4. Training configuration

**Key parameters:**
- `LEARNING_RATE = 2e-5` — was `1e-4` in the original run (caused NaN weights)
- `DISTRIBUTION = "tf_idf"` — matches the original training command
- `TEACHER_MODEL = STUDENT_MODEL = "distiluse-base-multilingual-cased-v2"` — same as original

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
TEACHER_MODEL = "distiluse-base-multilingual-cased-v2"
STUDENT_MODEL = "distiluse-base-multilingual-cased-v2"  # fine-tuned to become VI student

TEACHER_LANG  = "en"
STUDENT_LANG  = "vi"

# ── Training hyperparameters ───────────────────────────────────────────────────
DISTRIBUTION  = "tf_idf"   # options: tf_idf | padded_uniform | roberta
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5       # FIXED: was 1e-4, which caused NaN weight explosion
EPSILON       = 1.0        # OT regularization
EPOCHS        = 1          # increase to 2–4 for better quality if time allows

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Teacher : {TEACHER_MODEL}  (lang={TEACHER_LANG})")
print(f"Student : {STUDENT_MODEL}  (lang={STUDENT_LANG})")
print(f"Distribution  : {DISTRIBUTION}")
print(f"Learning rate : {LEARNING_RATE}  (FIXED from 1e-4)")
print(f"Epochs        : {EPOCHS}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Device        : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 5. Pre-create output directories

`KnowledgeDistillation.train()` uses string concatenation (no `os.makedirs` inside),
so we must create the save directory before training.

In [ ]:
STUDENT_SAVE_DIR = os.path.join(OUTPUT_DIR, f"sentence_transformer_multilingual_{DISTRIBUTION}")
os.makedirs(STUDENT_SAVE_DIR, exist_ok=True)

print("Student model will be saved to:", STUDENT_SAVE_DIR)

## 6. Train

In [ ]:
from models.knowledge_distillation import KnowledgeDistillation

trainer = KnowledgeDistillation(
    teacher_model_language = TEACHER_LANG,
    student_model_language = STUDENT_LANG,
    teacher_model          = TEACHER_MODEL,
    student_model          = STUDENT_MODEL,
    bitext_data            = BITEXT_CSV,
    save_dir               = OUTPUT_DIR,
    distribution           = DISTRIBUTION,
    device                 = DEVICE,
    batch_size             = BATCH_SIZE,
    learning_rate          = LEARNING_RATE,
    epsilon                = EPSILON,
    epochs                 = EPOCHS,
)

student_model_path = trainer.train()
print("\nTraining complete!")
print("Student model saved at:", student_model_path)

## 7. Sanity-check the saved model

Loads the student model and encodes a few Vietnamese sentences.  
Checks for NaN/zero-norm embeddings — if any are found, the model weights are still corrupted.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading saved student from:", student_model_path)
student = SentenceTransformer(student_model_path, device=DEVICE)

test_vi = [
    "xin vui lòng đặt người quét rác trong tủ chổi",
    "tình bạn bao gồm sự hiểu biết lẫn nhau",
    "im lặng một lát",
    "hãy để tôi kể cho các bạn về vài mảnh ghép nhé",
]

emb = student.encode(test_vi, convert_to_numpy=True, normalize_embeddings=False)

nan_count      = int(np.isnan(emb).sum())
inf_count      = int(np.isinf(emb).sum())
norms          = np.linalg.norm(emb, axis=1)
zero_norms     = int(np.sum(norms == 0))

print(f"\nEmbedding shape : {emb.shape}")
print(f"NaN count       : {nan_count}")
print(f"Inf count       : {inf_count}")
print(f"Zero-norm rows  : {zero_norms}")
print(f"Norm min/mean/max: {norms.min():.4f} / {np.nanmean(norms):.4f} / {norms.max():.4f}")

if nan_count == 0 and inf_count == 0 and zero_norms == 0:
    print("\n✅ Model is healthy — embeddings look good!")
else:
    print("\n❌ Model still has corrupted weights. Check training logs for NaN loss warnings.")

## 8. Quick bilingual retrieval evaluation

Encodes a small slice of the bitext with teacher (EN) and student (VI),
then computes positive/negative cosine scores and MRR.

In [ ]:
SEED = 42
EVAL_ROWS = 500   # use a small slice for fast eval

eval_df = pd.read_csv(BITEXT_CSV).head(EVAL_ROWS).reset_index(drop=True)
print(f"Evaluating on {len(eval_df)} sentence pairs...")

teacher = SentenceTransformer(TEACHER_MODEL, device=DEVICE)

src_emb_raw = teacher.encode(
    eval_df["source"].tolist(), batch_size=64,
    show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)

tgt_emb_raw = student.encode(
    eval_df["target"].tolist(), batch_size=64,
    show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)

# ── Safe L2 normalize (zero-norm → zero vector, not NaN) ─────────────────────
def safe_l2_normalize(emb):
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return emb / norms

src_emb = safe_l2_normalize(src_emb_raw)
tgt_emb = safe_l2_normalize(tgt_emb_raw)

# ── Positive cosine scores ────────────────────────────────────────────────────
pos_scores = np.sum(src_emb * tgt_emb, axis=1)

# ── Negative cosine scores (shuffled targets) ─────────────────────────────────
rng = np.random.default_rng(SEED)
neg_scores_all = []
for _ in range(5):
    perm = rng.permutation(len(src_emb))
    # ensure no identity mapping
    same = np.where(perm == np.arange(len(src_emb)))[0]
    for idx in same:
        swap = (idx + 1) % len(src_emb)
        perm[idx], perm[swap] = perm[swap], perm[idx]
    neg_scores_all.append(np.sum(src_emb * tgt_emb[perm], axis=1))
neg_scores = np.concatenate(neg_scores_all)

# ── Retrieval metrics (MRR, Acc@k) ────────────────────────────────────────────
n = len(src_emb)
sims = src_emb @ tgt_emb.T   # (n, n)
ranks = []
for i in range(n):
    correct = sims[i, i]
    if np.isnan(correct):
        ranks.append(n)
        continue
    ranks.append(int(np.nansum(sims[i] > correct) + 1))
ranks = np.array(ranks)

safe_mean = lambda a: float(np.nanmean(a)) if not np.all(np.isnan(a)) else float('nan')

print("\n" + "="*50)
print(f"positive_cosine_mean : {safe_mean(pos_scores):.4f}")
print(f"negative_cosine_mean : {safe_mean(neg_scores):.4f}")
print(f"margin_mean          : {safe_mean(pos_scores) - safe_mean(neg_scores):.4f}")
print(f"MRR                  : {np.mean(1.0 / ranks):.4f}")
print(f"Acc@1                : {np.mean(ranks <= 1):.4f}")
print(f"Acc@5                : {np.mean(ranks <= 5):.4f}")
print(f"Acc@10               : {np.mean(ranks <= 10):.4f}")
print("="*50)

## 9. Package output for download

In [ ]:
import subprocess

zip_path = f"/kaggle/working/phase1_model_{DISTRIBUTION}.zip"
subprocess.run(
    ["zip", "-r", zip_path, student_model_path],
    check=True
)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"Zipped model → {zip_path}  ({size_mb:.1f} MB)")
print("Done. Save this Kaggle session output as a Dataset to use in Phase 2 or evaluation.")

## 10. List all output files

In [ ]:
for root, dirs, files in os.walk("/kaggle/working"):
    level  = root.replace("/kaggle/working", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f"{indent}  {f}  ({size_mb:.2f} MB)")